# Phase 3: OCR Engines (EasyOCR & Tesseract Fallback)

**Goal:** Evaluate primary OCR (EasyOCR on GPU) combined with a fallback engine (Tesseract) on preprocessed prescription images. Measure confidence, speed, and establish the "verify manually" threshold for low-confidence or handwritten inputs.

## 0. Setup Cell

In [ ]:
# --- SETUP CELL ---
import os, sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / ".git").exists() and root != root.parent:
    root = root.parent
if not (root / ".git").exists():
    root = Path.cwd().resolve()
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repo root:", root)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 1. Load Data Index & Ground Truth

In [ ]:
import time, glob
import numpy as np
import pandas as pd
from PIL import Image

import src.data as D
import src.preprocess as P
import src.ocr as O
import importlib
importlib.reload(D)
importlib.reload(P)
importlib.reload(O)

from src.data import DATA_ROOT, build_image_index, open_image, gold_records

syn_index = build_image_index(os.path.join(DATA_ROOT, "synthetic"))
dev_gold_path = "data/dev_gold.csv"
gold_dev = {}
if os.path.exists(dev_gold_path):
    gold_dev = gold_records(pd.read_csv(dev_gold_path))
    dev_files = sorted(gold_dev.keys())
else:
    dev_files = sorted(syn_index.keys())[:20]

print(f"Found {len(dev_files)} dev images for OCR evaluation.")

## 2. Test OCR on Printed Dev Images

In [ ]:
if dev_files:
    test_fn = dev_files[0]
    img = open_image(test_fn, syn_index)
    stages = P.preprocess(img)
    
    print(f"Running OCR on {test_fn} (stage: denoised)...")
    out = O.ocr_with_fallback(stages["denoised"])
    
    print(f"\nEngine: {out['engine']} | Confidence: {out['conf']:.2f} | Time: {out['secs']:.2f} s")
    print("--- Raw Extracted Text ---")
    print(out["text"])
    if test_fn in gold_dev:
        print("--- Ground Truth Records ---")
        print(gold_dev[test_fn])
else:
    print("No dev images found. Please run notebooks/01_data.ipynb first.")

## 3. Test OCR on Handwritten Pages
Tests images placed in `~/ocr_data/handwritten_pages/`.

In [ ]:
hw_dir = os.path.join(DATA_ROOT, "handwritten_pages")
hw_images = glob.glob(f"{hw_dir}/*.*")
print("Handwritten images found:", len(hw_images))

for p in hw_images[:2]:
    hw_img = Image.open(p).convert("RGB")
    stages = P.preprocess(hw_img)
    out = O.ocr_with_fallback(stages["denoised"])
    print(f"File: {os.path.basename(p)} | Engine: {out['engine']} | Conf: {out['conf']:.2f} | Secs: {out['secs']:.2f}")
    print("Text excerpt:", repr(out["text"][:100]))

## 4. Compare Preprocessing Stages for Speed and OCR Confidence
Compare `gray`, `denoised`, and `binary` across a subset of dev images.

In [ ]:
eval_count = min(10, len(dev_files))
if eval_count > 0:
    comparison_rows = []
    for stage in ["gray", "denoised", "binary"]:
        confs, secs = [], []
        for i in range(eval_count):
            img = open_image(dev_files[i], syn_index)
            st = P.preprocess(img)[stage]
            res = O.run_easyocr(st)
            confs.append(res["conf"])
            secs.append(res["secs"])
        comparison_rows.append({
            "stage": stage,
            "avg_conf": float(np.mean(confs)),
            "avg_secs": float(np.mean(secs))
        })
    stage_summary = pd.DataFrame(comparison_rows)
    print(stage_summary.to_string(index=False))
else:
    print("No dev images available to run stage comparison.")

## 5. Confidence Threshold Determination for Verification Warning
- Printed prescriptions typically achieve confidence > 0.70.
- Handwritten prescriptions or poor quality photos generally drop below 0.55.
- Recommended default threshold: `conf < 0.55` triggers the warning: *"Low OCR confidence (possibly handwritten). Please verify manually."*